### Libraries

In [71]:
%pip install pandas scikit-learn pyarrow


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.




[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\medri\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [72]:
# pip install pandas
import pandas as pd
# pip install scikit-learn
from sklearn.model_selection import GroupShuffleSplit
# https://scikit-learn.org/0.20/modules/generated/sklearn.preprocessing.OrdinalEncoder.html
from sklearn.preprocessing import OrdinalEncoder
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html
from sklearn.naive_bayes import CategoricalNB

### Load Dataset

In [73]:
df = pd.read_parquet("data/lax_flights.parquet")

### Turn Dataset into Bayes Classifier Dataset

In [74]:
# https://pandas.pydata.org/docs/user_guide/timeseries.html
# turn each formal date into a day of the week (such as sun, tue, mon, etc.)
flight_date = pd.to_datetime(df.flightDate)
df["day"] = flight_date.dt.day_name().str[:3]

# calculate the booking window the buyer has. if the buy has a flight date of April 16, and searched for the ticket on April 14, then we do 16-14 = 2
# dt.days turns the number into a plain number (such as 2)
days_ahead = (flight_date - pd.to_datetime(df.searchDate)).dt.days

# https://pandas.pydata.org/docs/reference/api/pandas.cut.html
# bin each flight window the user has. this is for easier data reading.
df["window"] = pd.cut(days_ahead, bins=[0, 7, 21, 45, 60], labels=["1-7", "8-21", "22-45", "46-60"])

# function! sections off all hours into buckets so that we can see each hour as morning, afternoon, or night
def time_bucket(hour):
    if 3 <= hour <= 11: return "Morning"
    if 12 <= hour <= 17: return "Afternoon"
    return "Evening"

# departure string breakdown
# full departure time string
text = df.segmentsDepartureTimeRaw
# we want the hour only
hour_text = text.str[11:13]
# turn into an int
hour = hour_text.astype(int)
# use function above to morning, night, or afternoon
# call function like this bc we need to pass 2 MILLION pieces of data to it
df["time"] = hour.map(time_bucket)

# if there is a stop, 1+, if not, nonstop
df["stops"] = df.isNonStop.map({True: "Nonstop", False: "1+"})

# only keep first 2 chars of the airline (AB || CD turns into AB)
df["airline"] = df.segmentsAirlineCode.str[:2]

# display first 5 rows
df[["day", "time", "window", "stops", "airline"]].head()



,day,time,window,stops,airline
0,Sun,Evening,1-7,1+,F9
1,Sun,Evening,1-7,Nonstop,NK
2,Sun,Afternoon,1-7,Nonstop,NK
3,Sun,Morning,1-7,1+,B6
4,Sun,Afternoon,1-7,1+,UA


### Good Deal vs Bad Deal Training

In [75]:
# 20% of flights go into testing, random_state = 42 makes sure the state is the same random state each time (this is just for consistency, it's kinda like a seed), n_splits = 1 is how many diff train/test splits we make (1)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=3)

# split our dataset, making sure all rows with the same legID as one group that has to stay tgther
train_idx, test_idx = next(splitter.split(df, groups=df.legId))

# make the train and test tables using the rows we just calculated
# .copy makes it so that we have separate tables instead of just views into df
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# shared flights should be 0 if data was split correctly
print(len(train), len(test), "shared flights:", len(set(train.legId) & set(test.legId)))

1606182 406255 shared flights: 0


In [76]:
# cheapest-third cutoff for each destination, from TRAINING data only
# groups by airport type (such as BOS or ATL), and only looks at the fares
cutoff = train.groupby("destinationAirport").totalFare.quantile(0.33)

# good deal = fare at or below its route's cutoff
train["good_deal"] = train.totalFare <= train.destinationAirport.map(cutoff)
test["good_deal"] = test.totalFare <= test.destinationAirport.map(cutoff)

print(cutoff)
# check how many flights in the training dataset are good deals
print("amt/percentage of good deals:", train.good_deal.mean())

destinationAirport
ATL    328.6
BOS    327.1
LGA    327.2
Name: totalFare, dtype: float64
amt/percentage of good deals: 0.3305254323607163


### Printing out Bayes Classifier Table

In [77]:
# without random state, it would all be sunday if we printed just .head()
train[["day", "time", "window", "stops", "airline", "good_deal"]].sample(14, random_state=1)

,day,time,window,stops,airline,good_deal
1287237,Mon,Morning,8-21,1+,DL,False
77725,Thu,Morning,8-21,1+,UA,False
1361447,Sat,Afternoon,8-21,1+,AA,True
13301,Mon,Evening,46-60,1+,AA,False
1435547,Sat,Morning,22-45,1+,AA,True
1545898,Fri,Evening,1-7,1+,AA,True
1795917,Sun,Morning,8-21,1+,AA,False
193866,Thu,Morning,46-60,1+,UA,False
1409850,Thu,Evening,22-45,1+,B6,True
620630,Thu,Afternoon,22-45,1+,AA,False


In [78]:
good_deal_true = train[train["good_deal"]]
good_deal_true
 

,legId,searchDate,flightDate,startingAirport,destinationAirport,isNonStop,totalFare,segmentsDepartureTimeRaw,segmentsAirlineCode,day,window,time,stops,airline,good_deal
1,ac2edb3ea2542898a8e93ba21f8f21f1,2022-04-16,2022-04-17,LAX,ATL,True,328.59,2022-04-17T22:50:00.000-07:00,NK,Sun,1-7,Evening,Nonstop,NK,True
2,c87e7ad7cb1175da004234de4a377946,2022-04-16,2022-04-17,LAX,ATL,True,328.59,2022-04-17T17:20:00.000-07:00,NK,Sun,1-7,Afternoon,Nonstop,NK,True
92,013eef54d9f045be7b5d522a6b79886d,2022-04-16,2022-04-17,LAX,BOS,False,247.19,2022-04-17T17:45:00.000-07:00||2022-04-18T06:5...,SY||SY,Sun,1-7,Afternoon,1+,SY,True
93,c8bb1d46f43bbb61361a617d456d360d,2022-04-16,2022-04-17,LAX,BOS,False,311.60,2022-04-17T06:33:00.000-07:00||2022-04-17T08:3...,AA||AA,Sun,1-7,Morning,1+,AA,True
94,6b4ac449d0f31b38a361d7c2ed21d734,2022-04-16,2022-04-17,LAX,BOS,False,311.60,2022-04-17T13:27:00.000-07:00||2022-04-17T17:4...,UA||UA,Sun,1-7,Afternoon,1+,UA,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012417,b8c33965752f7eb4383ec3ca01d40ba1,2022-10-05,2022-11-11,LAX,LGA,False,297.20,2022-11-11T05:30:00.000-08:00||2022-11-11T16:1...,UA||UA,Fri,22-45,Morning,1+,UA,True
2012418,777cc4cd15bfedc451fef416e38bf130,2022-10-05,2022-11-11,LAX,LGA,False,297.60,2022-11-11T21:30:00.000-08:00||2022-11-12T06:4...,AA||AA,Fri,22-45,Evening,1+,AA,True
2012419,8aa3b3a9eafb6a42529716b1b8431153,2022-10-05,2022-11-11,LAX,LGA,False,308.60,2022-11-11T05:30:00.000-08:00||2022-11-11T11:2...,UA||UA,Fri,22-45,Morning,1+,UA,True
2012420,1b0cdafea8bbba70be34c5e0a0f885e8,2022-10-05,2022-11-11,LAX,LGA,False,308.60,2022-11-11T22:30:00.000-08:00||2022-11-12T08:3...,DL||DL,Fri,22-45,Evening,1+,DL,True


In [79]:
good_deal_false = train[~train["good_deal"]]
good_deal_false

,legId,searchDate,flightDate,startingAirport,destinationAirport,isNonStop,totalFare,segmentsDepartureTimeRaw,segmentsAirlineCode,day,window,time,stops,airline,good_deal
5,b5e5fa70deaf3eae8884e5e3bc45733f,2022-04-16,2022-04-17,LAX,ATL,False,357.60,2022-04-17T12:51:00.000-07:00||2022-04-17T17:4...,UA||UA,Sun,1-7,Afternoon,1+,UA,False
8,4f3368f7d550dd8a6bbd4e2c0209eecf,2022-04-16,2022-04-17,LAX,ATL,False,357.60,2022-04-17T00:04:00.000-07:00||2022-04-17T07:3...,UA||UA,Sun,1-7,Evening,1+,UA,False
9,f0d69d984c72865d89a0b2259e73f71f,2022-04-16,2022-04-17,LAX,ATL,False,357.60,2022-04-17T11:14:00.000-07:00||2022-04-17T17:4...,UA||UA,Sun,1-7,Morning,1+,UA,False
12,52711fb23380d7dcede2fabf357bbacb,2022-04-16,2022-04-17,LAX,ATL,False,369.10,2022-04-17T11:50:00.000-07:00||2022-04-17T20:4...,AA||AA,Sun,1-7,Morning,1+,AA,False
14,69d0c0cfff83b1024824925c5c1004a8,2022-04-16,2022-04-17,LAX,ATL,False,370.60,2022-04-17T14:51:00.000-07:00||2022-04-17T20:4...,AA||AA,Sun,1-7,Afternoon,1+,AA,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012429,43d190fc7f64f48ea90ad14d2a96bf57,2022-10-05,2022-11-11,LAX,LGA,False,373.57,2022-11-11T22:37:00.000-08:00||2022-11-12T14:2...,F9||F9,Fri,22-45,Evening,1+,F9,False
2012430,73438c8a3185bf87d40477e9ece77567,2022-10-05,2022-11-11,LAX,LGA,False,388.60,2022-11-11T05:30:00.000-08:00||2022-11-11T09:5...,UA||UA,Fri,22-45,Morning,1+,UA,False
2012431,5a5c16de80117e86d4e61888860bcf51,2022-10-05,2022-11-11,LAX,LGA,False,436.60,2022-11-11T08:40:00.000-08:00||2022-11-11T18:4...,AS||UA,Fri,22-45,Morning,1+,AS,False
2012432,767659dcde2d12392cbb79f849276ed2,2022-10-05,2022-11-11,LAX,LGA,False,437.60,2022-11-11T00:59:00.000-08:00||2022-11-11T07:1...,AA||AA,Fri,22-45,Evening,1+,AA,False


In [80]:
true_ratio = len(good_deal_true)/len(train)
true_ratio

0.3305254323607163

In [81]:
false_ratio = len(good_deal_false)/len(train)
false_ratio

0.6694745676392837

In [82]:
good_deal_days = good_deal_true.value_counts("day", normalize = True)
good_deal_days

day
Tue    0.197985
Wed    0.169860
Sat    0.153936
Mon    0.147134
Fri    0.128894
Thu    0.107312
Sun    0.094879
Name: proportion, dtype: float64

In [83]:
bad_deal_days = good_deal_false.value_counts("day", normalize = True)
bad_deal_days

day
Thu    0.162949
Sun    0.162897
Fri    0.150887
Mon    0.143663
Wed    0.132202
Sat    0.129082
Tue    0.118320
Name: proportion, dtype: float64

In [84]:
good_deal_departures = good_deal_true.value_counts("time", normalize = True)
good_deal_departures

time
Morning      0.443492
Evening      0.346045
Afternoon    0.210462
Name: proportion, dtype: float64

In [85]:
bad_deal_departures = good_deal_false.value_counts("time", normalize = True)
bad_deal_departures

time
Morning      0.480402
Evening      0.339086
Afternoon    0.180512
Name: proportion, dtype: float64